# 07 — H₂ = 0 Persistent Homology Verification

> *Large-N results were generated with the proprietary Isomorphic Engine; this notebook reproduces the analysis and figures from the released data.*

This notebook verifies the key topological claim: the spacing manifold of Riemann zeta zeros has **H₂ = 0** (no 2-dimensional voids) across all tested scales.

Two datasets:
1. **Scaling verification** (`h2_scaling_verification.json`) — Betti numbers from N=10³ to height ~10²²
2. **Extended Vietoris–Rips** (`h2_extended_results.json`) — Persistent homology via ripser/gudhi in ℝ³ sliding window

Physical interpretation: H₂ = 0 confirms the spacing manifold has the topology of geodesics on a 2D surface, consistent with the spectral operator H_D acting on C²³ ⊗ L²([0,2π]).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.facecolor': '#0A2540',
    'axes.facecolor': '#0A2540',
    'axes.edgecolor': '#D4AF37',
    'axes.labelcolor': 'white',
    'text.color': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'figure.figsize': (12, 5),
    'font.size': 12
})

## 1. Load data

In [ ]:
with open('../data/h2-topology/h2_scaling_verification.json') as f:
    scaling = json.load(f)

with open('../data/h2-topology/h2_extended_results.json') as f:
    extended = json.load(f)

print(f"Scaling verification: {scaling['scales_tested']} scales, verified={scaling['verified']}")
print(f"  Conclusion: {scaling['conclusion']}")
print(f"\nExtended VR: {len(extended['results'])} scales")
print(f"  Method: {extended['method']}")
print(f"  Library: {extended['library']}")
print(f"  Embedding: {extended['embedding']}")

## 2. Betti numbers across scales (scaling verification)

In [ ]:
results = scaling['results']
labels = [r['label'] if 'label' in r else f"N={r.get('n_zeros', '?')}" for r in results]
h0 = [r['h0'] for r in results]
h1 = [r['h1'] for r in results]
h2 = [r['h2'] for r in results]

# Print table
print(f"{'Scale':<25} {'H₀':>8} {'H₁':>10} {'H₂':>6}")
print('-' * 52)
for i, r in enumerate(results):
    label = labels[i]
    print(f"{label:<25} {r['h0']:>8} {r['h1']:>10} {r['h2']:>6}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(results))
short_labels = [f"$10^{{{i+3}}}$" if i < 4 else f"h~$10^{{{[12,20,22][i-4]}}}$" for i in range(len(results))]

# Left: H0 and H1
ax = axes[0]
ax.bar([i - 0.15 for i in x], h0, width=0.3, color='#66B2FF', label='H₀', alpha=0.8)
ax2 = ax.twinx()
ax2.bar([i + 0.15 for i in x], h1, width=0.3, color='#D4AF37', label='H₁', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(short_labels)
ax.set_ylabel('H₀ (components)', color='#66B2FF')
ax2.set_ylabel('H₁ (loops)', color='#D4AF37')
ax2.tick_params(axis='y', colors='#D4AF37')
ax.set_title('Betti Numbers H₀, H₁ Across Scales')
h0_patch = mpatches.Patch(color='#66B2FF', label='H₀')
h1_patch = mpatches.Patch(color='#D4AF37', label='H₁')
ax.legend(handles=[h0_patch, h1_patch], facecolor='#0A2540', edgecolor='#D4AF37', loc='upper left')

# Right: H2 = 0 verification
ax = axes[1]
ax.bar(x, h2, color='#FF4444', width=0.5, alpha=0.8)
ax.axhline(y=0, color='#D4AF37', linestyle='--', lw=2, alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(short_labels)
ax.set_ylabel('H₂ (voids)')
ax.set_title('H₂ = 0 Verified at All Scales', color='#D4AF37')
ax.set_ylim(-0.5, 1)
ax.text(3, 0.7, 'H₂ ≡ 0 ✓', fontsize=20, color='#D4AF37', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/h2_scaling_verification.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Vietoris–Rips persistent homology (extended results)

In [ ]:
ext = extended['results']
n_zeros = [r['n_zeros'] for r in ext]
ext_h0 = [r['h0'] for r in ext]
ext_h0f = [r['h0_final'] for r in ext]
ext_h1 = [r['h1'] for r in ext]
ext_h2 = [r['h2'] for r in ext]
ext_h2_life = [r['h2_max_life'] for r in ext]

print(f"{'N zeros':>8} {'H₀ init':>8} {'H₀ final':>9} {'H₁':>6} {'H₂':>4} {'H₂ max life':>12}")
print('-' * 52)
for r in ext:
    print(f"{r['n_zeros']:>8} {r['h0']:>8} {r['h0_final']:>9} {r['h1']:>6} {r['h2']:>4} {r['h2_max_life']:>12.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: H1 growth (shows linear scaling)
ax = axes[0]
ax.plot(n_zeros, ext_h1, 'o-', color='#D4AF37', lw=2, markersize=8)
ax.set_xlabel('Number of zeros')
ax.set_ylabel('H₁ (1-cycles / loops)')
ax.set_title('H₁ Grows Linearly with N')

# Fit line
coeffs = np.polyfit(n_zeros, ext_h1, 1)
fit_x = np.linspace(min(n_zeros), max(n_zeros), 100)
ax.plot(fit_x, np.polyval(coeffs, fit_x), '--', color='white', alpha=0.5, label=f'slope={coeffs[0]:.2f}')
ax.legend(facecolor='#0A2540', edgecolor='#D4AF37')

# Right: Persistence diagram (H2 max lifetime)
ax = axes[1]
ax.bar(range(len(n_zeros)), ext_h2_life, color='#FF4444', alpha=0.8, width=0.5)
ax.axhline(y=0.1, color='#D4AF37', linestyle='--', lw=1, alpha=0.6, label='noise threshold (0.1)')
ax.set_xticks(range(len(n_zeros)))
ax.set_xticklabels([str(n) for n in n_zeros])
ax.set_xlabel('Number of zeros')
ax.set_ylabel('H₂ max lifetime')
ax.set_title('H₂ Lifetimes: All Below Noise Threshold')
ax.legend(facecolor='#0A2540', edgecolor='#D4AF37')

plt.tight_layout()
plt.savefig('../figures/h2_vietoris_rips.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Summary

| Method | Scales | H₂ Result | Interpretation |
|--------|--------|-----------|----------------|
| Connectivity-based Betti | 7 (10³ to ~10²²) | H₂ = 0 everywhere | No 2-voids at any scale |
| Vietoris–Rips (ripser/gudhi) | 7 (100 to 1000) | H₂ = 0, max lifetime < 0.1 | Transient noise only |

**Conclusion:** The spacing manifold of Riemann zeta zeros exhibits **H₂ ≡ 0** at all tested scales, confirming the 1-dimensional geodesic structure on a 2D surface predicted by the spectral operator $H_D$ on $\mathbb{C}^{23} \otimes L^2([0,2\pi])$.